# Multi-Asset Portfolio Risk

A concise end-to-end risk review for a six-asset portfolio. The data is synthetic and deterministic, so the analysis is fully reproducible.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from portfolio_risk.config import ASSET_CLASSES, CONFIDENCE_LEVEL, RISK_LIMITS, WEIGHTS
from portfolio_risk.factor_risk import default_factor_exposures, factor_return_proxies
from portfolio_risk.market_data import generate_market_data
from portfolio_risk.pnl_attribution import factor_pnl_explain, risk_attribution
from portfolio_risk.portfolio import asset_class_exposure, portfolio_returns
from portfolio_risk.risk_models import compare_risk_models
from portfolio_risk.risk_metrics import rolling_risk_metrics
from portfolio_risk.risk_monitor import latest_alerts, risk_summary
from portfolio_risk.stress_testing import historical_scenarios, reverse_stress, run_stress_tests

sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:,.2%}'.format

## Portfolio and returns

In [ ]:
asset_returns = generate_market_data()
portfolio_pnl = portfolio_returns(asset_returns, WEIGHTS)

portfolio = pd.DataFrame({'weight': WEIGHTS, 'asset_class': ASSET_CLASSES})
display(portfolio)
display(asset_class_exposure(WEIGHTS, ASSET_CLASSES).to_frame())

## Risk limits

Historical risk measures are compared with the portfolio limits.

In [ ]:
summary = risk_summary(portfolio_pnl, RISK_LIMITS)
rolling = rolling_risk_metrics(portfolio_pnl, confidence=CONFIDENCE_LEVEL)

display(summary)
display(latest_alerts(rolling, RISK_LIMITS))

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
(1 + portfolio_pnl).cumprod().plot(ax=axes[0], color='#003e6b', title='Portfolio cumulative return')
axes[0].set_ylabel('Growth of £1')
rolling[['annualised_volatility', 'var_95', 'es_95']].plot(ax=axes[1], title='Rolling 63-day risk metrics')
axes[1].axhline(RISK_LIMITS['annualised_volatility'], color='tab:red', ls='--', label='Volatility limit')
axes[1].legend()
plt.tight_layout()

## Risk-model comparison

The same one-day risk is measured using historical simulation, a normal model, EWMA volatility, and Monte Carlo simulation.

In [ ]:
model_comparison = compare_risk_models(portfolio_pnl, CONFIDENCE_LEVEL)
display(model_comparison)

model_comparison.plot.bar(figsize=(9, 4), title='One-day VaR and Expected Shortfall (95%)')
plt.ylabel('Loss')
plt.xticks(rotation=0)
plt.tight_layout()

## P&L explain and risk attribution

Fixed factor exposures turn simple asset-return proxies into an explainable factor P&L. The residual is the part not captured by those factors.

In [ ]:
exposures = default_factor_exposures(asset_returns.columns)
factor_returns = factor_return_proxies(asset_returns)
pnl_explain, factor_contributions = factor_pnl_explain(asset_returns, WEIGHTS, exposures, factor_returns)
risk_contributions = risk_attribution(asset_returns, WEIGHTS)

display(risk_contributions)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
risk_contributions['component_var'].sort_values().plot.barh(ax=axes[0], color='#337ab7', title='Component VaR')
pnl_explain[['actual_pnl', 'factor_pnl', 'residual_pnl']].cumsum().plot(ax=axes[1], title='Cumulative P&L explain')
axes[1].set_ylabel('Cumulative return')
plt.tight_layout()

## Historical scenarios and reverse stress

The historical scenarios are illustrative single-period representations of recognisable market events. Reverse stress shows how much each shock would need to scale to reach the portfolio drawdown limit.

In [ ]:
scenarios = historical_scenarios()
stress_results = run_stress_tests(WEIGHTS, scenarios)
reverse_results = reverse_stress(WEIGHTS, scenarios, RISK_LIMITS['max_drawdown'])

display(stress_results)
display(reverse_results)

stress_results['Total'].plot.barh(figsize=(8, 3), color='#c44e52', title='Historical scenario portfolio P&L')
plt.xlabel('Portfolio return')
plt.tight_layout()